#### Data Loading & Proccesing

In [4]:
from huggingface_hub import dataset_info

info = dataset_info("TommyChien/UltraDomain")
print(info)

DatasetInfo(id='TommyChien/UltraDomain', author='TommyChien', sha='aa8a51d523f8fc3c5a0ab90dd16b7f6b9dbb5d0d', created_at=datetime.datetime(2024, 9, 6, 7, 56, 52, tzinfo=datetime.timezone.utc), last_modified=datetime.datetime(2024, 9, 9, 2, 48, 23, tzinfo=datetime.timezone.utc), private=False, gated=False, disabled=False, downloads=797, downloads_all_time=None, likes=47, paperswithcode_id=None, tags=['task_categories:question-answering', 'language:en', 'license:apache-2.0', 'region:us'], trending_score=None, card_data={'annotations_creators': None, 'language_creators': None, 'language': ['en'], 'license': 'apache-2.0', 'multilinguality': None, 'size_categories': None, 'source_datasets': None, 'task_categories': ['question-answering'], 'task_ids': None, 'paperswithcode_id': None, 'pretty_name': None, 'config_names': None, 'train_eval_index': None}, siblings=[RepoSibling(rfilename='.gitattributes', size=None, blob_id=None, lfs=None), RepoSibling(rfilename='README.md', size=None, blob_id=N

In [7]:
from huggingface_hub import hf_hub_download, list_repo_files
import json
import os

# Carpeta donde guardar todo
DATASET_DIR = "../../data/raw/ultradomain"
os.makedirs(DATASET_DIR, exist_ok=True)

# Lista de todos los dominios
dominios = [
    "agriculture", "art", "biography", "biology", "cooking", "cs",
    "fiction", "fin", "health", "history", "legal", "literature",
    "mathematics", "mix", "music", "philosophy", "physics", 
    "politics", "psychology", "technology"
]

# Descargar todos los jsonl
for dominio in dominios:
    print(f"Descargando {dominio}...")
    path = hf_hub_download(
        repo_id="TommyChien/UltraDomain",
        filename=f"{dominio}.jsonl",
        repo_type="dataset",
        local_dir=DATASET_DIR
    )
    print(f"  ✅ {path}")

print("🎉 Dataset completo descargado")

Descargando agriculture...
  ✅ ../../data/raw/ultradomain/agriculture.jsonl
Descargando art...
  ✅ ../../data/raw/ultradomain/art.jsonl
Descargando biography...
  ✅ ../../data/raw/ultradomain/biography.jsonl
Descargando biology...
  ✅ ../../data/raw/ultradomain/biology.jsonl
Descargando cooking...
  ✅ ../../data/raw/ultradomain/cooking.jsonl
Descargando cs...
  ✅ ../../data/raw/ultradomain/cs.jsonl
Descargando fiction...
  ✅ ../../data/raw/ultradomain/fiction.jsonl
Descargando fin...
  ✅ ../../data/raw/ultradomain/fin.jsonl
Descargando health...
  ✅ ../../data/raw/ultradomain/health.jsonl
Descargando history...
  ✅ ../../data/raw/ultradomain/history.jsonl
Descargando legal...
  ✅ ../../data/raw/ultradomain/legal.jsonl
Descargando literature...
  ✅ ../../data/raw/ultradomain/literature.jsonl
Descargando mathematics...
  ✅ ../../data/raw/ultradomain/mathematics.jsonl
Descargando mix...
  ✅ ../../data/raw/ultradomain/mix.jsonl
Descargando music...
  ✅ ../../data/raw/ultradomain/music.json

In [18]:
def preparar_dataset(dominio, n_libros=None):
    """Separa libros únicos y sus Q&A asociadas"""
    path = f"{DATASET_DIR}/{dominio}.jsonl"
    
    libros = {}   # context_id → {context, meta}
    qas = []      # lista de Q&A con referencia al libro
    
    with open(path, "r", encoding="utf-8") as f:
        for linea in f:
            item = json.loads(linea)
            
            # Guardar libro único (evita duplicados)
            cid = item["context_id"]
            if cid not in libros:
                if n_libros and len(libros) >= n_libros:
                    continue
                libros[cid] = {
                    "context": item["context"],
                    "meta": item["meta"],
                    "label": item["label"]
                }
            
            # Guardar Q&A
            qas.append({
                "id": item["_id"],
                "context_id": cid,
                "question": item["input"],
                "answer": item["answers"][0] if item["answers"] else "",
                "titulo": item["meta"]["title"]
            })
    
    return libros, qas


# Ejemplo: 2 libros de cs con todas sus preguntas
libros, qas = preparar_dataset("cs", n_libros=2)

print(f"Libros únicos: {len(libros)}")
print(f"Total Q&A: {len(qas)}")
for cid, libro in libros.items():
    n_preguntas = sum(1 for q in qas if q["context_id"] == cid)
    print(f"  📖 {libro['meta']['title']} → {n_preguntas} preguntas")

Libros únicos: 2
Total Q&A: 17
  📖 Machine Learning With Spark → 8 preguntas
  📖 Probability and Statistics for Computer Science → 9 preguntas


In [19]:
# Contar sin límite
libros_todos, qas_todas = preparar_dataset("cs", n_libros=None)
print(f"Total libros en cs: {len(libros_todos)}")
print(f"Total Q&A en cs: {len(qas_todas)}")
for cid, libro in libros_todos.items():
    n_preguntas = sum(1 for q in qas_todas if q["context_id"] == cid)
    print(f"  📖 {libro['meta']['title']} → {n_preguntas} preguntas")

Total libros en cs: 10
Total Q&A en cs: 100
  📖 Machine Learning With Spark → 8 preguntas
  📖 Probability and Statistics for Computer Science → 9 preguntas
  📖 Linux Kernel Networking → 8 preguntas
  📖 Modern Optimization With R → 9 preguntas
  📖 Guide to Java → 8 preguntas
  📖 Introducing Regular Expressions → 12 preguntas
  📖 Mastering VBA for Microsoft Office 2013 → 15 preguntas
  📖 Joe Celko's SQL Programming Style → 11 preguntas
  📖 Professional Microsoft SQL Server 2008 Programming → 11 preguntas
  📖 Introduction to the Theory of Programming Languages → 9 preguntas


In [17]:
dominios_validos = [d for d in dominios if d not in ["fin", "legal", "mix"]]

for dominio in dominios_validos:
    libros_d, qas_d = preparar_dataset(dominio, n_libros=None)
    print(f"{dominio:15} → {len(libros_d):3} libros | {len(qas_d):4} Q&A")

agriculture     →  12 libros |  100 Q&A
art             →  29 libros |  200 Q&A
biography       →  28 libros |  180 Q&A
biology         →  27 libros |  220 Q&A
cooking         →  14 libros |  120 Q&A
cs              →  10 libros |  100 Q&A
fiction         →  30 libros |  220 Q&A
health          →  26 libros |  180 Q&A
history         →  26 libros |  180 Q&A
literature      →  26 libros |  180 Q&A
mathematics     →  20 libros |  160 Q&A
music           →  29 libros |  200 Q&A
philosophy      →  26 libros |  200 Q&A
physics         →  19 libros |  160 Q&A
politics        →  25 libros |  180 Q&A
psychology      →  27 libros |  200 Q&A
technology      →  28 libros |  240 Q&A


In [20]:
import os
import json

PROCESSED_DIR = "../../data/processed/ultradomain"

def exportar_dominio(dominio):
    # Carpetas
    os.makedirs(f"{PROCESSED_DIR}/{dominio}", exist_ok=True)
    os.makedirs(f"{PROCESSED_DIR}/qa", exist_ok=True)
    
    libros, qas = preparar_dataset(dominio, n_libros=None)
    
    # Un .txt por libro
    for cid, libro in libros.items():
        txt_path = f"{PROCESSED_DIR}/{dominio}/{cid}.txt"
        with open(txt_path, "w", encoding="utf-8") as f:
            # Cabecera con metadatos
            f.write(f"Title: {libro['meta']['title']}\n")
            f.write(f"Authors: {libro['meta']['authors']}\n")
            f.write(f"Domain: {libro['label']}\n")
            f.write("="*80 + "\n\n")
            f.write(libro["context"])
    
    # Q&A en un json por dominio
    qa_path = f"{PROCESSED_DIR}/qa/{dominio}.json"
    with open(qa_path, "w", encoding="utf-8") as f:
        json.dump(qas, f, ensure_ascii=False, indent=2)
    
    print(f"✅ {dominio}: {len(libros)} libros, {len(qas)} Q&A")

# Exportar todos los dominios válidos
for dominio in dominios_validos:
    exportar_dominio(dominio)

✅ agriculture: 12 libros, 100 Q&A
✅ art: 29 libros, 200 Q&A
✅ biography: 28 libros, 180 Q&A
✅ biology: 27 libros, 220 Q&A
✅ cooking: 14 libros, 120 Q&A
✅ cs: 10 libros, 100 Q&A
✅ fiction: 30 libros, 220 Q&A
✅ health: 26 libros, 180 Q&A
✅ history: 26 libros, 180 Q&A
✅ literature: 26 libros, 180 Q&A
✅ mathematics: 20 libros, 160 Q&A
✅ music: 29 libros, 200 Q&A
✅ philosophy: 26 libros, 200 Q&A
✅ physics: 19 libros, 160 Q&A
✅ politics: 25 libros, 180 Q&A
✅ psychology: 27 libros, 200 Q&A
✅ technology: 28 libros, 240 Q&A


### Final function

In [24]:
import sys
sys.path.append("../../src")
from ultradomain import cargar_experimento, DOMINIOS_VALIDOS

libros, qas = cargar_experimento("biology", n_libros=1)

📚 Dominio: biology
   📖 The Serengeti Rules — Sean B. Carroll (11 preguntas)
   Total Q&A: 11
